In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# I-visualize ang timing ng circuit

<Accordion>
<AccordionItem title="Mga bersyon ng package">

Ang code sa pahinang ito ay binuo gamit ang mga sumusunod na kinakailangan.
Inirerekumenda naming gamitin ang mga bersyong ito o mas bago.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Habang ang [timeline drawer](/guides/visualize-circuit-timing) na built-in sa Qiskit ay kapaki-pakinabang para sa mga static na circuit, maaaring hindi nito tumpak na mailarawan ang timing ng mga [dynamic na circuit](/guides/classical-feedforward-and-control-flow) dahil sa mga implicit na operasyon tulad ng broadcasting at pagtukoy ng branch. Bilang bahagi ng suporta sa dynamic na circuit, ang Qiskit Runtime ay nagbabalik ng tumpak na impormasyon sa timing ng circuit sa loob ng mga resulta ng job kapag hiniling.

> **Note:** - Ito ay isang experimental na function. Nasa preview release status ito at samakatuwid ay maaaring magbago.
> - Ang function na ito ay naaangkop lamang sa mga Sampler job ng Qiskit Runtime.
> - Kahit na ang kabuuang oras ng circuit ay ibinabalik sa "compilation" metadata, HINDI ito ang oras na ginagamit para sa billing (QPU time).
### I-enable ang pagkuha ng timing data
Para i-enable ang pagkuha ng timing data, itakda ang experimental na flag na `scheduler_timing` sa `True` kapag nagpapatakbo ng primitive job.

In [1]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### I-access ang timing data ng circuit
Kapag hiniling, ang timing data ng circuit para sa bawat PUB ay ibinabalik sa metadata ng resulta ng job, sa ilalim ng `["compilation"]["scheduler_timing"]["timing"]`. Ang field na ito ay naglalaman ng raw na impormasyon sa timing. Para ipakita ang impormasyon sa timing, gamitin ang built-in na visualization tool, tulad ng inilarawan sa seksyon ng [I-visualize ang mga timing](#visualize-timings).

Gamitin ang sumusunod na code para ma-access ang timing data ng circuit para sa unang PUB:

In [2]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### Unawain ang raw timing data
Habang ang pag-visualize ng timing data ng circuit sa pamamagitan ng paggamit ng `draw_circuit_schedule_timing` method ay ang pinaka-karaniwang use case, maaaring maging kapaki-pakinabang na maunawaan ang istruktura ng raw timing data na ibinabalik. Maaari itong makatulong sa iyo, halimbawa, para mag-extract ng impormasyon nang programmatically.

Ang timing data na ibinabalik sa `["compilation"]["scheduler_timing"]["timing"]` ay isang listahan ng mga string. Ang bawat string ay kumakatawan sa isang tagubilin sa ilang channel at pinaghiwalay ng comma sa mga sumusunod na uri ng data:

- `Branch` - Tinutukoy kung ang tagubilin ay nasa control flow (then / else) o isang pangunahing branch.
- `Instruction` - Ang gate at ang qubit na dapat gawin.
- `Channel` - Ang channel na ini-assign ng tagubilin. Maaari itong isa sa mga sumusunod:
      - `Qubit x` - Ang drive channel para sa qubit _x_.
      - `AWGRx_y` (arbitrary waveform generator readout) - Ginagamit ng mga readout channel para makipag-communicate kapag sinusukat ang mga qubit. Ang mga argumento na _x_ at _y_ ay tumutugma sa readout instrument ID at numero ng qubit, ayon sa pagkakasunod.
- `T0` - Ang oras ng pagsisimula ng tagubilin sa loob ng kumpletong schedule
- `Duration` - Ang tagal ng tagubilin, sa mga yunit ng _dt_ segundo, kung saan ang 1 dt = 1 scheduling cycle. Mahahanap mo ang halaga ng `dt` ng isang backend sa pamamagitan ng paggamit ng [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- `Pulse` - Ang uri ng pulse operation na ginagamit.

Halimbawa:

In [3]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### I-visualize ang mga timing
Sa `qiskit-ibm-runtime` v0.43.0 o mas bago, maaari mong i-visualize ang mga timing ng circuit. Para i-visualize ang mga timing, kailangan mo munang i-convert ang metadata ng resulta sa `fig` sa pamamagitan ng paggamit ng [`draw_circuit_schedule_timing` method](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26). Ang method na ito ay nagbabalik ng `plotly` figure, na maaari mong ipakita nang direkta, i-save sa isang file, o pareho. Para sa karagdagang impormasyon tungkol sa mga `plotly` command na gagamitin, tingnan ang [`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) at [`fig.write_image("<path.format>")`](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html).

In [4]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d8287kugbeec73alfbug (DONE)


![Ang pag-hover sa output ay nagpapakita ng impormasyon tulad ng simula, katapusan, at tagal.](../docs/images/guides/visualize-circuit-timing/image_1.svg 'Halimbawa ng nabuong figure')
#### Unawain ang nabuong figure
Ang imahe ng output ng timing data ng circuit na na-output ng `draw_circuit_schedule_timing` ay naghahatid ng sumusunod na impormasyon:

- Ang X axis ay oras sa mga yunit ng _dt_ segundo, kung saan ang 1 dt = 1 scheduling cycle. Mahahanap mo ang halaga ng `dt` ng isang backend sa pamamagitan ng paggamit ng [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- Ang Y axis ay ang channel (isipin ang mga channel bilang mga instrumento na nagpapalabas ng mga pulse).
    - `Receive channel` - Ang tanging channel na hindi instrumento sa sarili. Ito ay isang tagubilin na nilalaro sa lahat ng channel na bahagi ng isang proseso ng komunikasyon sa hub sa panahon na iyon.
    - `Qubit x` - Ang drive channel para sa qubit x.
    - `AWGRx_y` (arbitrary waveform generator readout) - Ginagamit ng mga readout channel para makipag-communicate kapag sinusukat ang mga qubit. Ang mga argumento na _x_ at _y_ ay tumutugma sa readout instrument ID at numero ng qubit, ayon sa pagkakasunod.
    - `Hub` - Kinokontrol ang broadcasting.

Bukod dito, ang bawat tagubilin ay may format na *X_Y*, kung saan ang *X* ay ang pangalan ng tagubilin at *Y* ay ang uri ng pulse. Ang uri na `play` ay nag-aaplay ng mga control pulse, at ang `capture` ay nagtatala ng estado ng qubit. Maaari ka ring mag-hover sa bawat tagubilin para makakuha ng higit pang detalye. Halimbawa, ang nakaraang figure ay nagpapakita ng control pulse para sa X gate na inilapat sa qubit 10 sa 1161 dt.
### End-to-end na halimbawa
Ang halimbawang ito ay nagpapakita kung paano i-enable ang opsyon, makuha ang impormasyon sa timing ng circuit mula sa metadata, at ipakita ito bilang imahe.

Una, i-set up ang environment, tukuyin ang mga circuit at i-convert ang mga ito sa mga ISA circuit, at tukuyin at patakbuhin ang mga job.

In [5]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,1365,0,shift_phase\nmain,sx_0,Qubit 0,1365,9,play\nmain,sx_0,Qubit 0,1369,0,shift_phase\nmain,rz_0,Qubit 0,1374,0,shift_phase\nmain,barrier,Qubit 0,1374,0,barrier\nmain,measure_0,Qubit 0,1374,64,play\nmain,measure_0,Qubit 0,1438,108,play\nmain,measure_0,AWGR0_0,1485,360,capture\nmain,measure_0,Qubit 0,1546,64,play\nmain,measure_0,Qubit 0,1610,64,play\nmain,barrier,Qubit 0,2046,0,barrier\nmain,broadcast,Hub,1485,561,broadcast\nmain,receive,Receive,2046,7,receive\nthen,x_0,Qubit 0,2061,9,play\nmain,barrier,Qubit 0,2079,0,barrier\nmain,measure_0,Qubit 0,2079,64,play\nmain,measure_0,Qubit 0,2143,108,play\nmain,measure_0,AWGR0_0,2190,360,capture\nmain,measure_0,Qubit 0,2251,64,play\nmain,measure_0,Qubit 0,2315,64,play\nmain,barrier,Qubit 0,2725,0,barrier\nmain,barrier,Qubit 0,2725,0,barrier\n'

Finally, you can visualize and save the timing:

In [6]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

Susunod, makuha ang timing ng circuit schedule: